In [ ]:
import logging
import sys

from pydantic import ConfigDict

from wags_llm.client.bedrock import BedrockClaudeJsonClient
from wags_llm.services.structured_task import StructuredTaskRunner

logging.basicConfig(
    stream=sys.stdout,
    level=logging.WARNING,
    format="%(name)s - %(levelname)s - %(message)s",
)
logging.getLogger("wags_llm").setLevel(logging.DEBUG)

In [19]:
from collections.abc import Mapping
from typing import Any

from wags_llm.skills.base import BaseSkillTemplate


class VariantCurationSkill(BaseSkillTemplate):
    skill_path = "skills/variant_curation.md"
    version = "0.1.0"

    def build_user_prompt(self, payload: Mapping[str, Any]) -> str:
        variant = payload["variant"]
        disease = payload.get("disease", "cancer")

        return f"""Curate the following variant for {disease}.

Variant:
{variant}

Return concise JSON matching the provided schema:
- clinical_significance: one short sentence
- evidence_level: short label
- supporting_rationale: 2-3 short sentences maximum
"""

from pydantic import BaseModel

class VariantCurationResult(BaseModel):
    model_config = ConfigDict(extra="forbid", use_enum_values=True)  # Required

    clinical_significance: str | None = None
    evidence_level: str | None = None
    supporting_rationale: str | None = None
    error_message: str | None = None

In [22]:
registry = SkillRegistry()
registry.register(VariantCurationSkill())

MODEL_ID = "us.anthropic.claude-sonnet-4-6"
REGION_NAME = "us-east-1"
PROFILE_NAME = "dev-account"
MAX_TOKENS = 350

llm_client = BedrockClaudeJsonClient(
    model_id=MODEL_ID,
    region_name=REGION_NAME,
    profile_name=PROFILE_NAME,
    max_tokens=MAX_TOKENS,
    temperature=0,
)

task_runner = StructuredTaskRunner(
    client=llm_client,
    skill_registry=registry,
)

result = task_runner.execute_skill(
    skill_name="variant_curation",
    skill_version="0.1.0",
    payload={
        "variant": "BRAF V600E",
        "disease": "melanoma",
    },
    response_model=VariantCurationResult,
)

wags_llm.skills.registry - DEBUG - Registering skill: name='variant_curation', version='0.1.0'
wags_llm.client.bedrock - DEBUG - BedrockClaudeJsonClient config: model_id='us.anthropic.claude-sonnet-4-6', region_name='us-east-1', profile_name='dev-account', max_tokens=350, temperature=0.000000
wags_llm.client.bedrock - INFO - BedrockClaudeJsonClient successfully initialized for model_id='us.anthropic.claude-sonnet-4-6'
wags_llm.skills.base - DEBUG - Loading skill from path: skills/variant_curation.md
wags_llm.skills.base - INFO - Loaded skill from path: skills/variant_curation.md
wags_llm.client.bedrock - DEBUG - Bedrock Claude usage={'inputTokens': 500, 'outputTokens': 200, 'totalTokens': 700, 'cacheReadInputTokens': 0, 'cacheWriteInputTokens': 0}
wags_llm.client.bedrock - DEBUG - Bedrock Claude metrics={'latencyMs': 5032}
wags_llm.client.bedrock - DEBUG - Bedrock Claude content=[{'text': '{"clinical_significance":"BRAF V600E is a well-established oncogenic driver mutation in melanoma 